# Student Performance Prediction using Machine Learning
**Week 6 Final Assignment — Artificial Intelligence**

**Objective:** Predict a student's final score from study habits, attendance, prior performance, and support indicators.

> **Dataset note:** The bundled CSV is a deterministic *illustrative/synthetic* dataset created for offline reproducibility and for demonstrating cleaning (missing values and duplicate rows). It is not real student data. For a real-data version, see the optional UCI Student Performance dataset section near the end. Do not present synthetic results as results from real students.

## 1. Problem definition
This is a **regression** task because the target (`final_score`) is numeric. The model estimates a score from 0–100. It is intended as a learning prototype, not as a high-stakes tool for grading or decisions about individual students.

### Workflow
Problem definition → data loading → cleaning → EDA → feature/target selection → train/test split → preprocessing pipeline → model training → prediction → evaluation → conclusion.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

## 2. Load and understand the dataset

In [ ]:
# The CSV is supplied alongside this notebook. Keep it in the same folder.
df = pd.read_csv("student_performance_dataset.csv")

print("First five records:")
display(df.head())
print("Last five records:")
display(df.tail())
print("Shape (rows, columns):", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nSummary statistics:")
display(df.describe(include="all").T)

## 3. Data quality checks and preprocessing

In [ ]:
print("Missing values before cleaning:")
display(df.isna().sum().to_frame("missing_count"))
print("Duplicate rows before cleaning:", df.duplicated().sum())

before_rows = len(df)
data = df.drop_duplicates().copy()

numeric_cols_all = data.select_dtypes(include=np.number).columns
categorical_cols_all = data.select_dtypes(exclude=np.number).columns

# Demonstration cleaning: median for numeric fields, mode for categorical fields.
# In a production workflow, fit imputers only on the training data to avoid leakage.
for col in numeric_cols_all:
    data[col] = data[col].fillna(data[col].median())
for col in categorical_cols_all:
    mode_values = data[col].mode()
    if not mode_values.empty:
        data[col] = data[col].fillna(mode_values.iloc[0])

print(f"Rows before duplicate removal: {before_rows}")
print(f"Rows after duplicate removal: {len(data)}")
print("\nMissing values after cleaning:")
display(data.isna().sum().to_frame("missing_count"))
display(data.head())

### Features and target
- **Target (`y`):** `final_score`
- **Numerical features:** study hours, attendance, previous score, assignments completed, sleep hours
- **Categorical features:** internet access, extracurricular participation, parental support

The bundled dataset is synthetic. Its features are illustrative, and any patterns are not evidence about actual students.

## 4. Exploratory Data Analysis (EDA): at least four visualizations

In [ ]:
sns.set_theme()

In [ ]:
# Visualization 1: Distribution of final scores
plt.figure(figsize=(7, 4))
sns.histplot(data=data, x="final_score", bins=15, kde=True)
plt.title("Distribution of Final Scores")
plt.xlabel("Final score")
plt.ylabel("Number of students (synthetic records)")
plt.tight_layout()
plt.show()

**Insight:** This histogram shows how the target scores are distributed and whether the sample has a central cluster, skew, or extreme values. Because the dataset is synthetic, the shape reflects the data-generation process.

In [ ]:
# Visualization 2: Average score by parental support
plt.figure(figsize=(7, 4))
sns.barplot(data=data, x="parental_support", y="final_score", errorbar=None, order=["Low", "Medium", "High"])
plt.title("Average Final Score by Parental Support")
plt.xlabel("Parental support")
plt.ylabel("Mean final score")
plt.tight_layout()
plt.show()

**Insight:** The chart compares average scores across support categories. It is descriptive and does not prove that parental support causes a change in grades.

In [ ]:
# Visualization 3: Study hours vs final score
plt.figure(figsize=(7, 4))
sns.scatterplot(data=data, x="study_hours_per_day", y="final_score", hue="internet_access", alpha=0.75)
plt.title("Study Hours vs Final Score")
plt.xlabel("Study hours per day")
plt.ylabel("Final score")
plt.tight_layout()
plt.show()

**Insight:** The scatter plot helps assess whether score and study time move together and whether the pattern varies by internet access. Overlap and spread remind us that one feature alone cannot explain all outcomes.

In [ ]:
# Visualization 4: Correlation heatmap for numeric variables
plt.figure(figsize=(8, 5))
corr = data.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

**Insight:** Correlation summarizes linear associations between numeric variables. Correlation is not causation, and correlations from synthetic data should not be generalized to real learners.

In [ ]:
# Optional Visualization 5: Attendance vs final score
plt.figure(figsize=(7, 4))
sns.regplot(data=data, x="attendance_percent", y="final_score", scatter_kws={"alpha": 0.5}, line_kws={"color": "black"})
plt.title("Attendance vs Final Score")
plt.xlabel("Attendance (%)")
plt.ylabel("Final score")
plt.tight_layout()
plt.show()

## 5. Model development
A **Random Forest Regressor** is selected because it can learn non-linear relationships and feature interactions, works with mixed feature types after encoding, and usually requires less feature scaling sensitivity than linear models. Its limitations include reduced interpretability and the risk of overfitting on small or unrepresentative datasets.

We use a held-out test set (20%). Preprocessing is placed inside a scikit-learn pipeline so imputers, scaling, and category encoding are fitted using training data only. This is safer than fitting preprocessing on the complete dataset.

In [ ]:
TARGET = "final_score"
X = data.drop(columns=[TARGET])
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=250, min_samples_leaf=2, random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 6. Prediction examples

In [ ]:
results = pd.DataFrame({
    "actual_score": y_test.to_numpy(),
    "predicted_score": np.round(predictions, 1)
})
display(results.head(10))

## 7. Model evaluation (regression metrics)

In [ ]:
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

evaluation = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R²"],
    "Value": [mae, mse, rmse, r2],
    "Meaning": [
        "Average absolute prediction error, in score points",
        "Average squared error; penalizes larger errors",
        "Square root of MSE, in score points",
        "Proportion of target variance explained relative to the test-set mean baseline"
    ]
})
display(evaluation)

In [ ]:
# Actual vs predicted plot
plt.figure(figsize=(6, 5))
sns.scatterplot(x=y_test, y=predictions)
plt.plot([0, 100], [0, 100], linestyle="--", color="black")
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.title("Actual vs Predicted Scores")
plt.xlabel("Actual score")
plt.ylabel("Predicted score")
plt.tight_layout()
plt.show()

# Residual plot
residuals = y_test.to_numpy() - predictions
plt.figure(figsize=(7, 4))
sns.scatterplot(x=predictions, y=residuals)
plt.axhline(0, linestyle="--", color="black")
plt.title("Residuals vs Predicted Scores")
plt.xlabel("Predicted score")
plt.ylabel("Residual (actual - predicted)")
plt.tight_layout()
plt.show()

### Interpreting the metrics
- **MAE:** Typical absolute error in score points; lower is better.
- **MSE:** Mean squared error; lower is better, but it is expressed in squared score units.
- **RMSE:** Error in the original score units; more sensitive to large errors than MAE.
- **R²:** Relative fit against predicting the test-set mean. It can be negative if the model performs worse than that baseline.

Metrics depend on the dataset, split, and random seed. Do not compare synthetic-data metrics directly with results from real student records.

## 8. Limitations, ethics, and future scope
1. The bundled data is synthetic and cannot establish real-world predictive validity.
2. Real student outcomes are influenced by factors not represented here, including teaching context, health, accessibility, and assessment design.
3. Use consented, anonymized, appropriately governed data; minimize sensitive features and check subgroup performance.
4. Predictions should support early academic assistance, not label students or determine high-stakes outcomes.
5. Future work: evaluate on a documented real dataset, compare Random Forest with Linear Regression and Gradient Boosting, use cross-validation and hyperparameter tuning, analyze feature importance responsibly, and monitor model drift.

## 9. Optional real-data extension (UCI Student Performance)
The UCI Student Performance dataset includes school, demographic, social, and grade-related attributes for Portuguese secondary-school students. You can obtain it from the UCI Machine Learning Repository: https://archive.ics.uci.edu/dataset/320/student+performance

For a real-data extension, use `student-mat.csv` and predict `G3`. If the intended use is prediction *before* the final grading period, exclude `G1` and `G2` because they are earlier grades and may make the task easier in a way that is not available at the intended prediction time; state clearly which features are available at prediction time. Never merge this dataset with the synthetic sample and report the result as one dataset.

## 10. Conclusion
This notebook demonstrates a complete regression workflow: loading and inspecting data, handling missing values and duplicates, EDA, a leakage-aware preprocessing pipeline, model training, prediction, and regression evaluation. The bundled results are a reproducible demonstration on synthetic data. A credible real-world conclusion requires validation on an appropriate real dataset and careful consideration of fairness, privacy, and intended use.